### RaschPy Differential Item Functioning (DIF) worked example

Differential Item Functioning occurs when persons with the same overall trait level, but from different groups (e.g. Gender, first language), have different probabilities of success on a given item — a threat to fair measurement. `dif_test()` (available on `SLM`, `PCM`, and `RSM` — not yet implemented for `MFRM`) tests every item for DIF against a chosen reference group:

1. Splits non-extreme persons into groups by a named column in `self.exogenous` (any number of groups; each "focal" group is compared to a single "reference" group, by default the largest).
2. Calibrates each group independently, then purifies both groups onto a common scale — genuinely DIF-affected items are excluded from the scale used to *detect* DIF, so they can't distort their own test.
3. Tests every item (not just the ones used to define the purified scale) via a per-item Wald test, likelihood-ratio test, or both, plus an omnibus test (is there DIF at all, jointly across every item).

For PCM and RSM (which have a threshold/category structure in addition to item location), `dif_test()` also separately tests for Differential *Step* Functioning (DSF) — the same item location, but a shifted category structure — via `threshold_dif_table`.

This notebook simulates two groups (A and B) with known, deliberately injected DIF, and checks whether `dif_test()` recovers it.

#### SLM

In [1]:
import numpy as np
import pandas as pd
from raschpy import SLM, PCM, RSM
from raschpy.simulation import SLM_Sim, PCM_Sim, RSM_Sim

Simulate two groups of 300 persons each on the same 12 items. Group B has genuine DIF injected on three items: Item_1 (+1.0 logits, harder for B), Item_2 (-1.0 logits, easier for B), Item_3 (+0.5 logits). All other items are unaffected.

In [2]:
n = 300
sim_a = SLM_Sim(no_of_items=12, no_of_persons=n, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5
sim_b = SLM_Sim(no_of_items=12, no_of_persons=n, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(n)]
resp_b.index = [f'B_{i}' for i in range(n)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * n + ['B'] * n}, index=responses.index)

slm = SLM(responses, exogenous=exogenous)

Run the DIF test, comparing group B against reference group A:

In [3]:
slm.dif_test(
    'Group',
    reference='A',
    selection_method='wald',   # purification method: 'wald' (default) | 'robust_z' | 'none'
    test='both',               # per-item flagging: 'wald' | 'lr' | 'both'
    omnibus=True,              # Andersen-style joint test, all items at once
    category=True,             # adds an ETS-style A/B/C DIF-magnitude category column
    correction='bh',           # multiple-comparison correction: 'bh' | 'bonferroni' | None
    no_of_samples=300,
)

round(slm.dif_table, 3)

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,
Item_1,B,0.190,1.060,1.070,0.880,0.199,4.427,0.000,0.000,False,True,C+,18.479,0.000,0.000,True
Item_2,B,0.905,-0.154,-0.144,-1.048,0.192,-5.465,0.000,0.000,False,True,C-,24.764,0.000,0.000,True
Item_3,B,-0.197,0.388,0.398,0.595,0.189,3.142,0.002,0.007,False,True,B+,10.710,0.001,0.004,True
Item_4,B,0.592,0.279,0.289,-0.303,0.196,-1.550,0.121,0.364,False,False,A,2.402,0.121,0.364,False
Item_5,B,-1.236,-1.036,-1.026,0.210,0.213,0.985,0.325,0.779,True,False,A,0.767,0.381,0.762,False
Item_6,B,-0.303,-0.383,-0.373,-0.070,0.189,-0.369,0.712,0.831,True,False,A,0.186,0.666,0.875,False
Item_7,B,0.075,0.080,0.090,0.014,0.191,0.075,0.940,0.940,True,False,A,0.004,0.951,0.951,False
Item_8,B,-0.027,-0.095,-0.085,-0.059,0.180,-0.327,0.744,0.831,True,False,A,0.063,0.802,0.875,False
Item_9,B,-0.854,-0.773,-0.764,0.091,0.193,0.469,0.639,0.831,True,False,A,0.145,0.704,0.875,False


Items 1-3 (where DIF was injected) should be flagged, with `Difference` estimates close to the injected 1.0 / -1.0 / 0.5 logit shifts; the remaining items should not be flagged.

In [4]:
slm.dif_omnibus_table  # is there DIF at all, across every item jointly

,LR,df,p,Flagged
B,59.000587,11,0.0,True


#### PCM

Simulate two groups on 10 five-category items. Group B gets the same item-location DIF pattern as before (Item_1/2/3), plus a pure category-structure (DSF) shift on Item_4 — its central item location is left unchanged, but its threshold spacing is not, so it should be flagged in `threshold_dif_table` rather than `dif_table`.

In [5]:
n = 1000
max_score_vector = [3] * 10
sim_a = PCM_Sim(no_of_items=10, no_of_persons=n, max_score_vector=max_score_vector, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

# Pin thresholds to sim_a's own for every item except Item_4, which gets a
# deliberate step-structure (DSF) shift with no item-location shift.
manual_thresholds = []
for item in sim_a.item_names:
    vals = sim_a.thresholds.loc[item].dropna().values.copy()
    if item == 'Item_4':
        vals[0] += 0.8
    vals[-1] = -vals[:-1].sum()  # exact zero-sum, avoiding float residue
    manual_thresholds.append(vals)

sim_b = PCM_Sim(no_of_items=10, no_of_persons=n, max_score_vector=max_score_vector, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=manual_thresholds, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(n)]
resp_b.index = [f'B_{i}' for i in range(n)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * n + ['B'] * n}, index=responses.index)

pcm = PCM(responses, max_score_vector=max_score_vector, exogenous=exogenous)

In [6]:
pcm.dif_test(
    'Group',
    reference='A',
    selection_method='wald',
    test='both',
    omnibus=True,
    omnibus_scope='full',
    category=True,
    correction='bh',
    no_of_samples=300,
)

round(pcm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,
Item_1,B,-0.988,0.109,0.190,1.178,0.072,16.362,0.000,0.000,False,True,C+,306.005,0.000,0.000,True
Item_2,B,-0.196,-1.300,-1.219,-1.023,0.071,-14.399,0.000,0.000,False,True,C-,231.168,0.000,0.000,True
Item_3,B,0.860,1.325,1.406,0.545,0.084,6.515,0.000,0.000,False,True,B+,61.735,0.000,0.000,True
Item_4,B,-0.561,-0.634,-0.553,0.009,0.064,0.139,0.890,0.890,True,False,A,0.078,0.781,0.781,False
Item_5,B,-0.093,-0.133,-0.052,0.041,0.062,0.660,0.509,0.636,True,False,A,0.362,0.547,0.760,False
Item_6,B,1.208,1.068,1.149,-0.060,0.083,-0.718,0.473,0.636,True,False,A,0.106,0.745,0.781,False
Item_7,B,-0.293,-0.199,-0.118,0.175,0.067,2.598,0.009,0.023,False,False,A,3.516,0.061,0.152,False
Item_8,B,-0.423,-0.422,-0.341,0.082,0.068,1.202,0.229,0.386,True,False,A,1.102,0.294,0.490,False
Item_9,B,1.266,1.088,1.169,-0.097,0.081,-1.196,0.232,0.386,True,False,A,2.191,0.139,0.278,False


In [7]:
round(pcm.threshold_dif_table, 3)   # per-item category-structure (DSF) test

Group  Reference  Focal  Difference     SE      z      p  \
Item    Category                                                            
Item_1  1            B      1.752  2.003       0.251  0.242  1.036  0.300   
        2            B      0.923  0.998       0.075  0.225  0.333  0.739   
Item_2  1            B      0.507  0.208      -0.299  0.266 -1.125  0.260   
        2            B      1.118  1.283       0.165  0.231  0.717  0.473   
Item_3  1            B      0.971  1.013       0.042  0.226  0.184  0.854   
        2            B      1.883  1.985       0.102  0.284  0.357  0.721   
Item_4  1            B      1.827  0.470      -1.357  0.255 -5.330  0.000   
        2            B      1.002  0.394      -0.608  0.240 -2.531  0.011   
Item_5  1            B      1.551  0.910      -0.641  0.237 -2.703  0.007   
        2            B      0.566  1.066       0.500  0.238  2.097  0.036   
Item_6  1            B      2.706  1.895      -0.811  0.223 -3.641  0.000   
        2            B      0.189  1.069       0.879  0.339  2.593  0.010   
Item_7  1            B      2.126  2.160       0.034  0.215  0.157  0.875   
        2            B      1.495  1.089      -0.406  0.216 -1.877  0.061   
Item_8  1            B      1.752  1.642      -0.109  0.240 -0.454  0.650   
        2            B      0.954  1.066       0.112  0.225  0.500  0.617   
Item_9  1            B      1.157  1.442       0.285  0.224  1.275  0.202   
        2            B      0.705  0.345      -0.360  0.303 -1.187  0.235   
Item_10 1            B      1.594  1.523      -0.071  0.283 -0.251  0.802   
        2            B      2.213  2.098      -0.115  0.218 -0.528  0.598   

                  p (corrected)  Flagged  
Item    Category                          
Item_1  1                 0.600    False  
        2                 0.739    False  
Item_2  1                 0.473    False  
        2                 0.473    False  
Item_3  1                 0.854    False  
        2                 0.854    False  
Item_4  1                 0.000     True  
        2                 0.011     True  
Item_5  1                 0.014     True  
        2                 0.036     True  
Item_6  1                 0.001     True  
        2                 0.010     True  
Item_7  1                 0.875    False  
        2                 0.121    False  
Item_8  1                 0.650    False  
        2                 0.650    False  
Item_9  1                 0.235    False  
        2                 0.235    False  
Item_10 1                 0.802    False  
        2                 0.802    False

Item_4 should be flagged in `threshold_dif_table` (its category structure differs between groups) but not in `dif_table` (its item location is unaffected) — the two phenomena are deliberately kept independent in this simulation so the signal is unambiguous.

#### RSM

Simulate two groups on 20 items sharing one rating scale. Group B again gets the Item_1/2/3 location DIF, plus a shift to the shared threshold vector — since RSM's thresholds are shared across all items, this is a test-wide DSF signal rather than a single-item one.

In [8]:
n = 300
sim_a = RSM_Sim(no_of_items=20, no_of_persons=n, max_score=3, item_range=2, seed=1)

items_b = sim_a.items.copy()
items_b['Item_1'] += 1.0
items_b['Item_2'] -= 1.0
items_b['Item_3'] += 0.5

thresholds_b = sim_a.thresholds.copy()
thresholds_b[1] -= 0.5
thresholds_b[3] += 1
thresholds_b -= thresholds_b.mean()

sim_b = RSM_Sim(no_of_items=20, no_of_persons=n, max_score=3, item_range=2,
                manual_items=items_b, manual_item_names=sim_a.item_names,
                manual_thresholds=thresholds_b, seed=2)

resp_a, resp_b = sim_a.responses.copy(), sim_b.responses.copy()
resp_a.index = [f'A_{i}' for i in range(n)]
resp_b.index = [f'B_{i}' for i in range(n)]
responses = pd.concat([resp_a, resp_b])
exogenous = pd.DataFrame({'Group': ['A'] * n + ['B'] * n}, index=responses.index)

rsm = RSM(responses, exogenous=exogenous)

In [9]:
rsm.dif_test(
    'Group',
    reference='A',
    selection_method='robust_z',
    test='both',
    omnibus=True,
    category=True,
    correction='bh',
    no_of_samples=300,
)

round(rsm.dif_table, 3)   # item-location DIF

,Group,Reference,Focal,Focal (purified),Difference,SE,z,p,p (corrected),Selected,Flagged,Category,LR,p_LR,p_LR (corrected),Flagged_LR
Item,,,,,,,,,,,,,,,,
Item_1,B,0.301,1.144,1.142,0.841,0.148,5.679,0.000,0.000,False,True,C+,67.740,0.000,0.000,True
Item_2,B,0.973,0.086,0.083,-0.889,0.140,-6.332,0.000,0.000,False,True,C-,73.441,0.000,0.000,True
Item_3,B,0.033,0.410,0.407,0.375,0.139,2.686,0.007,0.048,True,False,A,17.879,0.000,0.000,False
Item_4,B,0.859,0.941,0.939,0.080,0.146,0.547,0.585,0.865,True,False,A,1.787,0.181,0.604,False
Item_5,B,-0.814,-0.772,-0.775,0.040,0.151,0.262,0.793,0.881,True,False,A,0.295,0.587,1.000,False
Item_6,B,-0.095,-0.427,-0.430,-0.335,0.134,-2.500,0.012,0.062,True,False,A,1.917,0.166,0.604,False
Item_7,B,0.234,0.165,0.162,-0.072,0.130,-0.553,0.580,0.865,True,False,A,0.104,0.747,1.000,False
Item_8,B,0.355,0.284,0.282,-0.073,0.146,-0.501,0.616,0.865,True,False,A,1.201,0.273,0.780,False
Item_9,B,-0.570,-0.475,-0.477,0.093,0.135,0.686,0.493,0.865,True,False,A,0.000,1.000,1.000,False


In [10]:
rsm.dif_omnibus_table     # joint item-location test

,LR,df,p,Flagged
B,225.911458,19,0.0,True


In [11]:
round(rsm.threshold_dif_table, 3)   # shared threshold-vector DIF

,Group,Category,Reference,Focal,Difference,SE,z,p,p (corrected),Flagged
0,B,1,0.172,0.614,0.442,0.107,4.149,0.0,0.0,True
1,B,2,1.200,2.184,0.984,0.102,9.668,0.0,0.0,True


#### A simpler alternative: `andersen_lr_test(split_by='exogenous')`

For a quick two-group check without the full anchor-purification workflow (no per-item table, just a single joint test), use `andersen_lr_test(split_by='exogenous')`. Note `split_by='person_location'`/`'score'` — using a median split on person location or raw score as a general model-fit test — is disabled as of a 2026-07 simulation study that found it has no power to detect genuine misfit; `split_by='exogenous'` (group membership from real covariate data) is unaffected and fully supported.

In [12]:
slm.andersen_lr_test(split_by='exogenous', covariate='Group')
print(f'LR = {slm.andersen_lr:.2f}, df = {slm.andersen_df}, p = {slm.andersen_p:.4f}')

LR = 59.00, df = 11, p = 0.0000
